# fair-prep — Colab runner

End-to-end on Colab GPU (T4 / A100 / L4): SFT, DPO, cross-size LoRA transfer, merging scaling-law.

**Requirements**: Runtime → Change runtime type → Hardware: GPU (T4 free / A100 paid).

**VS Code integration**: install the *Google Colab* extension in VS Code, then open this notebook and pick a Colab kernel.

## 1 · Bootstrap

In [ ]:
# Clone + install deps + unsloth (CUDA-only)
import os
os.environ['WITH_UNSLOTH'] = '1'           # set '0' to skip unsloth
# os.environ['HF_TOKEN'] = 'hf_xxx'         # only if you need gated models

!bash <(curl -sL https://raw.githubusercontent.com/Shumatsurontek/fair-prep/main/colab/setup.sh)
%cd /content/fair-prep/lora-bench

In [ ]:
!./lb status
!./lb models

## 2 · Baseline eval (no adapter)

In [ ]:
!./lb eval gsm8k   -M qwen3-0.6b -n 200 -o results/baseline.json
!./lb eval math500 -M qwen3-0.6b -n 200 -o results/math500_baseline.json

## 3 · SFT (standard)

In [ ]:
!./lb train sft -M qwen3-0.6b -c configs/sft_gsm8k.yaml
!./lb eval gsm8k   -a runs/qwen3_06b_sft_gsm8k/final -n 200 -o results/sft.json
!./lb eval math500 -a runs/qwen3_06b_sft_gsm8k/final -n 200 -o results/math500_sft.json

## 4 · SFT (unsloth fast path, optional)

In [ ]:
# Same SFT but 2-4× faster + 50-70 % less VRAM via unsloth.
# Auto-fallback to standard TRL if unsloth not installed.
!./lb train sft -M qwen3-0.6b -c configs/sft_gsm8k.yaml --fast

## 5 · Cross-size LoRA transfer (research)

Teacher (Qwen3-4B) LoRA → student (Qwen3-0.6B) via OLS projection on calibration activations.

In [ ]:
# 5.1 download a public Qwen3-4B math LoRA (adapter only, ~120 MB)
!python -c "from huggingface_hub import snapshot_download; \
  snapshot_download('saaduddinM/MATH_SFT_Q3.0-4BI_r16', \
    local_dir='runs/teacher_4b/math_sft', \
    allow_patterns=['adapter_*','*.json'])"

In [ ]:
# 5.2 build calibration set from MATH-500 prompts (small)
!mkdir -p data && python -c "\
import json; from datasets import load_dataset; \
ds = load_dataset('HuggingFaceH4/MATH-500', split='test').shuffle(seed=42).select(range(256)); \
open('data/calib_math.jsonl','w').write('\n'.join(json.dumps({'text':e['problem']}) for e in ds))"

In [ ]:
# 5.3 project teacher → student.  CUDA = no Mac mem cap, use n_calib=256 max_length=512.
!./lb transfer \
    -T Qwen/Qwen3-4B-Instruct-2507 \
    -S Qwen/Qwen3-0.6B \
    -a runs/teacher_4b/math_sft \
    -C data/calib_math.jsonl \
    -o runs/transferred/math_sft \
    -d cuda -n 256

In [ ]:
# 5.4 eval transferred adapter
!./lb eval math500 -a runs/transferred/math_sft -n 200 -o results/math500_transferred.json

## 6 · Merging scaling law (paper: arXiv:2509.24244)

After training k teacher experts (one per math domain), project them all, merge, eval, fit `L(k) = L∞ + A/(k+b)`.

In [ ]:
# Example merge once multiple transferred adapters exist
!./lb merge -m ties \
    -a runs/transferred/algebra \
    -a runs/transferred/geometry \
    -a runs/transferred/number_theory \
    -o runs/merged_ties_k3
!./lb eval math500 -a runs/merged_ties_k3 -n 200 -o results/scaling/ties_k3.json

## 7 · Report

In [ ]:
!./lb report
from IPython.display import Markdown
Markdown(open('results/REPORT.md').read())